# 01_seed_clickstream_backfill

Backfills the Eventhouse `Clickstream` table with synthetic events covering the
same 90-day window as `00_seed_historical_data` (which only seeds SQL + ADLS).
Without this notebook the Eventhouse is empty until the Function App starts
emitting, leaving a 3-month gap that breaks silver/gold time-series analytics
on first run.

Writes via the **Kusto Spark connector** straight to the KQL DB (queued ingest).
Bypasses the Eventstream entirely. Idempotent: drops + extends rows whose
`event_id` is generated deterministically from the row index + a fixed seed.
Re-running produces the same `event_id`s so a `summarize by event_id` dedupe
is trivial if needed (but Kusto's queued ingest doesn't auto-dedupe; we use
`.set` semantics via `replace` write mode below to keep it clean for re-runs).

## Parameters

Injected by `deploy.ps1` via placeholder replacement (same pattern as the
seed notebook). Defaults below let the notebook be run interactively for
ad-hoc re-backfills.

In [ ]:
kusto_cluster_uri = ""          # e.g. https://trd-xxxx.z2.kusto.fabric.microsoft.com
kusto_database    = "contoso_retail_events"
kusto_table       = "Clickstream"

# Volume target across the 90-day window. ~50 events / order avg over 50k
# orders gives a defensible ~2.5M; we pick a middle-ground 2M to keep the
# Kusto Spark queued-ingest under a few minutes.
n_events     = 2_000_000
n_customers  = 5_000
n_products   = 1_500

import datetime as _dt
backfill_end   = _dt.date.today()
backfill_start = backfill_end - _dt.timedelta(days=90)

random_seed = 4242

## Generate events

Built entirely in Spark (no driver-side `range(n_events)` collect). Schema and
value distributions match the Function App emitter (`function_app.py`) so a
consumer reading the full `Clickstream` table sees a single coherent stream.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

start_unix = int(__import__('datetime').datetime.combine(backfill_start, __import__('datetime').time(0,0,0)).timestamp())
end_unix   = int(__import__('datetime').datetime.combine(backfill_end,   __import__('datetime').time(0,0,0)).timestamp())
window_sec = end_unix - start_unix

# Same weights as function_app.py::EVENT_TYPE_WEIGHTS / DEVICE_WEIGHTS
event_type_buckets = [
    (0.45, 'page_view'),
    (0.70, 'product_view'),
    (0.80, 'add_to_cart'),
    (0.84, 'remove_from_cart'),
    (0.90, 'checkout_start'),
    (0.94, 'checkout_complete'),
    (1.00, 'search'),
]
device_buckets = [
    (0.42, 'desktop'),
    (0.92, 'mobile'),
    (1.00, 'tablet'),
]
channels = ['organic','google_ads','meta_ads','email','direct','referral']

def _bucket_expr(col, buckets):
    # Build nested when() chain matching the cumulative weight buckets.
    expr = F.lit(buckets[-1][1])
    for threshold, label in reversed(buckets):
        expr = F.when(col < F.lit(threshold), F.lit(label)).otherwise(expr)
    return expr

df = spark.range(0, n_events).withColumnRenamed('id', 'idx')

# Deterministic per-row randomness via murmur3 + idx + seed -> 64 unsigned bits.
# Spark's rand() is non-deterministic between runs; we want re-runs to overlap.
df = df.withColumn('r1', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed))    )) % 1_000_000) / 1_000_000.0)
df = df.withColumn('r2', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed+1)))) % 1_000_000) / 1_000_000.0)
df = df.withColumn('r3', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed+2)))) % 1_000_000) / 1_000_000.0)
df = df.withColumn('r4', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed+3)))) % 1_000_000) / 1_000_000.0)
df = df.withColumn('r5', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed+4)))) % 1_000_000) / 1_000_000.0)
df = df.withColumn('r6', (F.abs(F.xxhash64(F.col('idx').cast('string'), F.lit(str(random_seed+5)))) % 1_000_000) / 1_000_000.0)

# Deterministic event_id: stable UUID-shaped hex from idx + seed. Not RFC 4122
# compliant but human-readable as a hex string the same way the Function uses uuid4().
df = df.withColumn('event_id',
    F.concat_ws('-',
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-id')), 256), 1, 8),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-id')), 256), 9, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-id')), 256), 13, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-id')), 256), 17, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-id')), 256), 21, 12),
    )
)
df = df.withColumn('session_id',
    F.concat_ws('-',
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-sess')), 256), 1, 8),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-sess')), 256), 9, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-sess')), 256), 13, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-sess')), 256), 17, 4),
        F.substring(F.sha2(F.concat(F.col('idx').cast('string'), F.lit(f'-{random_seed}-sess')), 256), 21, 12),
    )
)

# Spread evenly across the 90-day window (uniform; good enough for demo).
df = df.withColumn('event_ts',
    F.to_timestamp(F.from_unixtime(F.lit(start_unix) + (F.col('r1') * F.lit(window_sec)).cast('long')))
)
df = df.withColumn('event_type', _bucket_expr(F.col('r2'), event_type_buckets))
df = df.withColumn('device',     _bucket_expr(F.col('r3'), device_buckets))
df = df.withColumn('channel',
    F.element_at(F.array([F.lit(c) for c in channels]), ((F.col('r4') * F.lit(len(channels))).cast('int') + F.lit(1)))
)
df = df.withColumn('customer_id', ((F.col('r5') * F.lit(n_customers)).cast('long') + F.lit(1)))
df = df.withColumn('product_id',  ((F.col('r6') * F.lit(n_products)).cast('long')  + F.lit(1)))
df = df.withColumn('page_url', F.concat(F.lit('/p/'), F.col('product_id').cast('string')))

# Project to exact KQL Clickstream schema column order.
events = df.select(
    'event_id',
    F.col('event_ts').cast(TimestampType()).alias('event_ts'),
    'event_type',
    'customer_id',
    'product_id',
    'session_id',
    'device',
    'channel',
    'page_url',
)

print(f'Generated DataFrame: {events.count():,} rows over {backfill_start} -> {backfill_end}')

## Write to KQL via Kusto Spark connector

Authenticates as the running notebook user (Fabric notebook AAD token). For
deploy-time execution we'll inject the workspace identity, which already has
`ingestors` role on the KQL DB (granted by `Grant-FabricKqlDatabaseWorkspaceIdentityAccess`).

In [ ]:
if not kusto_cluster_uri:
    raise ValueError('kusto_cluster_uri parameter is required')

# Kusto Spark connector ships with Fabric Spark by default; no package install
# needed. AAD device-flow / interactive falls through to the notebook's own
# AAD token, which is what we want for an interactive POC run.
from notebookutils import mssparkutils  # type: ignore
kusto_token = mssparkutils.credentials.getToken(kusto_cluster_uri)

(events.write
    .format('com.microsoft.kusto.spark.synapse.datasource')
    .option('kustoCluster',  kusto_cluster_uri)
    .option('kustoDatabase', kusto_database)
    .option('kustoTable',    kusto_table)
    .option('accessToken',   kusto_token)
    .option('tableCreateOptions', 'FailIfNotExist')
    .mode('append')
    .save())

print('Backfill ingest submitted to KQL (queued ingestion completes async, ~1-3 min).')

## Verify ingestion

Polls the table count until it stabilizes (queued ingest is async).

In [ ]:
import time, requests
headers = { 'Authorization': f'Bearer {kusto_token}', 'Content-Type': 'application/json' }
url = f'{kusto_cluster_uri}/v1/rest/query'

def query(csl):
    r = requests.post(url, json={'db': kusto_database, 'csl': csl}, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()

for attempt in range(20):
    res = query(f'{kusto_table} | count')
    cnt = res['Tables'][0]['Rows'][0][0]
    print(f'  attempt {attempt+1}: row count = {cnt:,}')
    if cnt >= n_events * 0.95:
        break
    time.sleep(15)

# Show date histogram so we can see the 90-day window populated.
hist = query(f'{kusto_table} | summarize cnt=count() by bin(event_ts, 7d) | order by event_ts asc')
for row in hist['Tables'][0]['Rows']:
    print(f'  {row[0]}  {row[1]:>10,}')